In [1]:
# ── DEBUG: بررسی مستقیم توکن و chat_id ────────────────────────────────
import json, urllib.request, urllib.error

TOKEN   = "8313609248:AAH0i5z-0lJS7MVulFHKXf0IcWPSqKjh5FY"
CHAT_ID = "-1002383929199"

url = f"https://api.telegram.org/bot{TOKEN}/sendMessage"
payload = json.dumps({"chat_id": CHAT_ID, "text": "debug test"}).encode()
req = urllib.request.Request(url, data=payload, headers={"Content-Type": "application/json"})
try:
    with urllib.request.urlopen(req, timeout=10) as resp:
        print("SUCCESS:", resp.read().decode())
except urllib.error.HTTPError as e:
    print("HTTP ERROR:", e.code, e.read().decode())
except Exception as e:
    print("ERROR:", e)

SUCCESS: {"ok":true,"result":{"message_id":27483,"sender_chat":{"id":-1002383929199,"title":"BullishBearish","type":"channel"},"chat":{"id":-1002383929199,"title":"BullishBearish","type":"channel"},"date":1779389066,"text":"debug test"}}


In [2]:
# ── تست کامل Mt5Notifier — همه event-ها ──────────────────────────────
import sys
sys.path.insert(0, r"d:\bot\ema-1d trend\ema-h1trend")
from telegram_bot.mt5_notifier import Mt5Notifier

n = Mt5Notifier()
print(f"enabled: {n._enabled}")
print(f"chat_id: {n._chat_id}")
print(f"url:     {n._url[:50]}...")
print()

# signal
n.notify_signal({
    "direction": "BUY",
    "entry": 2341.50,
    "sl": 2338.20,
    "tp": 2348.10,
    "ob_time": "2024-01-15 10:00:00",
    "displaced_by": 4.2,
})
print("sent: signal")

# slippage_adjusted
n.notify_slippage_adjusted({
    "direction": "BUY",
    "ob_entry": 2341.50,
    "market_price": 2344.20,
    "slippage_pts": 2.7,
    "sl_unchanged": 2338.20,
    "volume_original": 0.02,
    "volume_adjusted": 0.01,
    "tp_original": 2348.10,
    "tp_adjusted": 2350.60,
})
print("sent: slippage_adjusted")

# order_placed
n.notify_order_placed({
    "ticket": 123456789,
    "direction": "BUY",
    "volume": 0.01,
    "sl": 2338.20,
    "tp": 2350.60,
    "slippage_pts": 2.7,
})
print("sent: order_placed")

# skip — slippage_exceeded
n.notify_skip({
    "reason": "slippage_exceeded",
    "direction": "BUY",
    "slippage_pts": 7.1,
    "max_pts": 6.0,
})
print("sent: skip(slippage_exceeded)")

# skip — position_open
n.notify_skip({
    "reason": "position_open",
    "open_positions": 1,
    "missed_signal": {
        "direction": "SELL",
        "entry": 2355.00,
        "sl": 2358.50,
        "tp": 2348.00,
    },
})
print("sent: skip(position_open)")

# position_closed — TP
n.notify_position_closed(ticket=123456789, profit=18.50, balance=1050.00, equity=1050.00)
print("sent: position_closed (TP)")

# position_closed — SL
n.notify_position_closed(ticket=987654321, profit=-9.25, balance=1040.75, equity=1040.75)
print("sent: position_closed (SL)")

print("\n✅ همه پیام‌ها ارسال شدند — کانال رو چک کن")

enabled: True
chat_id: -1002383929199
url:     https://api.telegram.org/bot8313609248:AAH0i5z-0lJ...

sent: signal
sent: slippage_adjusted
sent: order_placed
sent: skip(slippage_exceeded)
sent: skip(position_open)
sent: position_closed (TP)
sent: position_closed (SL)

✅ همه پیام‌ها ارسال شدند — کانال رو چک کن
